# 02: The Discriminator (the CNN that scores real vs. simulated)

**Goal:** understand the CNN inside pg-gan-mosquito that judges whether a genomic region looks real or fake. This is the second half of the GAN. Notebook 01 covered the first half (the simulator).

By the end of this notebook you should be able to:

1. Build the discriminator and inspect its architecture (~666K parameters).
2. Feed it a synthetic genotype matrix and read its output.
3. Explain **why** they built it with two population branches instead of one, and what the `(1, 5)` convolutional kernel does.
4. Understand what the CNN's single-number output means (a logit, not a probability).

No training, no real data. Just build, inspect, forward-pass.

**Kernel:** Python 3.11 (Mathieson)

## Cell 1: Setup

Same path detection as notebook 01. Also silences TensorFlow's chatty log messages.

In [ ]:
import sys, os, warnings
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # silence TF info/warning logs
warnings.filterwarnings('ignore')

# Locate pg-gan-mosquito (same logic as notebook 01)
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
CANDIDATES = [
    REPO_ROOT.parent / 'pg-gan-mosquito',
    REPO_ROOT.parent / 'Mathieson_Application' / 'pg-gan-mosquito',
]
PGGAN_REPO = next((p for p in CANDIDATES if p.exists()), None)
if PGGAN_REPO is None:
    raise FileNotFoundError('pg-gan-mosquito not found. Clone as sibling of this repo.')
sys.path.insert(0, str(PGGAN_REPO))

import numpy as np
import tensorflow as tf
import discriminator

print(f'TensorFlow version: {tf.__version__}')
print(f'pg-gan-mosquito loaded from: {PGGAN_REPO}')

## Cell 2: What we are building

**One-liner:** the discriminator is a CNN whose job is to look at a genomic region and predict *is this real mosquito data or is it something the simulator generated*?

**Slightly longer:**

During pg-gan training, this network sees a mixed batch of genomic regions. Some are real (pulled from the Ag1000G HDF5 file). Some are simulated (produced by `msprime` under whatever θ the generator is currently trying). The network outputs a single number per region: high = looks real, low = looks fake.

There are three model classes in `discriminator.py`, one for each supported population count:

- `OnePopModel`: single population.
- `TwoPopModel`: two populations. **This is what the paper uses** (POP1 = Guinea, POP2 = Burkina Faso).
- `ThreePopModel`: three populations.

We'll work with `TwoPopModel` throughout.

**A note on the terminology:** in machine learning, "discriminator" is the standard GAN term for the network that tries to catch fakes. It's the pop-gen equivalent of a Turing test judge (see notebook 01 or the paper for more on this framing).

## Cell 3: Build a TwoPopModel and count its parameters

**One-liner:** the model has ~666K trainable parameters. Not huge by modern ML standards (a small vision model might have 10M+, GPT-4 has ~1.7T), but plenty for this task.

Below we instantiate the model with the paper's exact population sizes: 62 GN haplotypes and 162 BF haplotypes. Then we force the model to build its internal weight tensors by calling it once with a dummy input (Keras subclassed models create weights lazily on first call, not at instantiation). Then we count the parameters layer by layer.

In [ ]:
# Paper's populations: 31 GN diploid = 62 haplotypes; 81 BF diploid = 162 haplotypes
POP1_HAPS = 62
POP2_HAPS = 162
NUM_SNPS = 72       # per-region SNP window
CHANNELS = 2        # channel 0: allele (0/1); channel 1: inter-SNP distances
BATCH_SIZE = 50

model = discriminator.TwoPopModel(pop1=POP1_HAPS, pop2=POP2_HAPS)

# Force weight creation with a dummy call
dummy = tf.zeros((BATCH_SIZE, POP1_HAPS + POP2_HAPS, NUM_SNPS, CHANNELS), dtype=tf.float32)
_ = model(dummy, training=False)

print(f'Total parameters:     {model.count_params():>10,}')
print(f'Trainable parameters: {sum(int(tf.size(w)) for w in model.trainable_variables):>10,}')
print()
print('Per-layer breakdown:')
print(f"  {'layer':<15} {'shape':<25} {'params':>15}")
print('  ' + '-' * 55)
layer_names = ['conv1 kernel', 'conv1 bias', 'conv2 kernel', 'conv2 bias',
               'fc1 kernel', 'fc1 bias', 'fc2 kernel', 'fc2 bias',
               'dense3 kernel', 'dense3 bias']
for name, w in zip(layer_names, model.trainable_variables):
    print(f'  {name:<15} {str(list(w.shape)):<25} {int(tf.size(w)):>15,}')

**Read the output:** notice that **the convolutional layers are tiny** (~10K params combined) while the first dense layer (`fc1`) owns ~614K parameters (~92% of the network's total). This is normal for a small CNN. The conv layers pattern-match small local features (a few SNPs at a time); the huge dense layer takes those pooled features and mixes them together into a single decision.

## Cell 4: The two-branch design (why POP1 and POP2 have separate pipelines)

**One-liner:** the CNN processes POP1 and POP2 through separate copies of the same convolutional layers, then combines their features at the end. This lets it learn population-specific patterns before it compares the two.

**Longer:** for `TwoPopModel`, the forward pass looks like this schematically:

```
Input tensor shape: (batch, 224, 72, 2)
                    (batch × individuals × SNPs × channels)
                    │
         ┌──────────┴───────────┐
         │                      │
         ▼                      ▼
   POP1 slice              POP2 slice
   x[:, :62, :, :]         x[:, 62:, :, :]
  (batch, 62, 72, 2)      (batch, 162, 72, 2)
         │                      │
     conv1 + pool           conv1 + pool         ← SAME layer applied to both
         │                      │
     conv2 + pool           conv2 + pool         ← same again
         │                      │
    ReduceMean over          ReduceMean over    
    individuals axis         individuals axis    ← permutation-invariant pool
         │                      │
      flatten                flatten             
         │                      │
         └──────── concat ──────┘
                    │
                    ▼
                FC1 (320 units)
                    │
                Dropout (0.5)
                    │
                FC2 (128 units)
                    │
                Dropout (0.5)
                    │
                Dense3 (1 unit) → single logit
```

**Why separate branches?** If you dumped all 224 haplotypes into one CNN, the network would have a hard time distinguishing "a POP1-specific pattern" from "a POP2-specific pattern" because they'd be smashed together in the same feature maps. By processing each population separately with the same convolutional filters, the CNN gets two feature summaries ("here's what POP1 looks like", "here's what POP2 looks like") and only combines them at the dense layers where the comparison actually happens.

It's the same idea as **Siamese networks** in computer vision, if you've heard of those: two parallel branches with shared weights, whose outputs get compared at the end.

## Cell 5: The `(1, 5)` kernel and why it preserves permutation invariance

**One-liner:** the convolutional kernel is `(1, 5)`, which means "1 individual tall, 5 SNPs wide". Each filter looks at 5 consecutive SNPs of a **single haplotype** at a time. The same filter is then slid across all haplotypes independently.

**Why it matters:** in the genotype matrix, the *order of haplotypes doesn't mean anything*. Haplotype #37 isn't more "meaningful" than haplotype #38. Rearranging rows should not change the output.

A traditional image CNN uses a kernel like `(3, 3)` that mixes information across both spatial dimensions. If you used that here, shuffling haplotype order would completely change the feature maps, and the CNN would try to memorize a particular ordering. Bad.

With a `(1, 5)` kernel, each haplotype's features are computed independently of every other haplotype. Then the permutation-invariant pool (next cell) collapses them into a summary. This guarantees the network's output doesn't depend on how the haplotypes were ordered.

**Verify it in code below.** We create a batch, then create a shuffled version (haplotypes reordered), and confirm the network's outputs match.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

# One batch of random genotype data
batch = tf.random.uniform((BATCH_SIZE, POP1_HAPS + POP2_HAPS, NUM_SNPS, CHANNELS),
                           minval=0, maxval=2, dtype=tf.int32)
batch = tf.cast(batch, tf.float32)

# Shuffle the haplotypes within each population (POP1 rows 0-61, POP2 rows 62-223)
shuffle_pop1 = np.random.permutation(POP1_HAPS)
shuffle_pop2 = np.random.permutation(POP2_HAPS) + POP1_HAPS
shuffled_indices = np.concatenate([shuffle_pop1, shuffle_pop2])
batch_shuffled = tf.gather(batch, shuffled_indices, axis=1)

# Run both through the discriminator
out_original = model(batch, training=False).numpy().flatten()
out_shuffled = model(batch_shuffled, training=False).numpy().flatten()

max_diff = np.abs(out_original - out_shuffled).max()
print(f'Max absolute difference in outputs: {max_diff:.2e}')
print(f'Original first 5 outputs:  {out_original[:5]}')
print(f'Shuffled first 5 outputs:  {out_shuffled[:5]}')

if max_diff < 1e-4:
    print('\n\u2705 Permutation-invariant confirmed: shuffling haplotypes did not change the output.')
else:
    print('\n\u274C Something is off (this should not happen for TwoPopModel).')

## Cell 6: ReduceMean, the permutation-invariant pool

**One-liner:** after the convolutional layers extract features per-haplotype, `ReduceMean` collapses across the haplotype dimension by taking the average. That average is permutation-invariant by definition (order of elements does not affect a mean).

This is why the previous cell's shuffle test worked. The whole architecture is deliberately designed so no downstream layer ever sees the haplotype ordering.

**Why mean instead of sum?** The paper explicitly changed this from `sum` (the original pg-gan default) to `mean` for the mosquito adaptation. Reason: POP1 has 62 haplotypes and POP2 has 162. If you summed, POP2's contribution to the feature vector would automatically be 2.6x larger than POP1's, purely because of sample size imbalance. Taking the mean normalizes each population to the same scale so the CNN sees them on equal footing.

**Alternatives** the paper's authors experimented with (visible as commented-out code in `discriminator.py`):

- `reduce_max`: take the maximum activation across haplotypes. Sensitive to outlier haplotypes.
- `reduce_sum`: what the original pg-gan used (works when populations have equal sample sizes).

They settled on `reduce_mean` for the two-population mosquito case.

In [ ]:
# Peek at the ReduceMean layer directly
reduce_mean = discriminator.ReduceMean()

# Toy example: 3 haplotypes, 4 SNPs, 2 filters
toy = tf.constant([[[[1., 2.], [3., 4.], [5., 6.], [7., 8.]],
                    [[9., 10.], [11., 12.], [13., 14.], [15., 16.]],
                    [[17., 18.], [19., 20.], [21., 22.], [23., 24.]]]])
print(f'Toy input shape:       {toy.shape}   (batch, haplotypes, SNPs, filters)')

reduced = reduce_mean(toy)
print(f'After ReduceMean:      {reduced.shape}   (batch, SNPs, filters)')
print(f'\nOriginal (per-haplotype values at SNP 0, filter 0): 1, 9, 17')
print(f'Mean of those:                                     {(1+9+17)/3}')
print(f'ReduceMean output at [batch=0, SNP=0, filter=0]:   {reduced[0, 0, 0].numpy()}')

## Cell 7: Forward pass through the whole model

**One-liner:** feed the model a genotype matrix, get a single number per region. That number is a **logit**, not a probability.

**What's a logit?** A raw pre-sigmoid score. The final layer of the network is `Dense(1)` with no activation function attached, so the output can be any real number (positive or negative, unbounded). To turn it into a probability between 0 and 1, apply the sigmoid function: `p = 1 / (1 + exp(-logit))`.

Why not just have the model output a probability directly? Numerical stability during training. Computing binary cross-entropy loss directly from logits (via `tf.keras.losses.BinaryCrossentropy(from_logits=True)`) is more numerically robust than computing sigmoid first and then taking the log. This is standard practice in modern deep learning.

**Interpretation guide:**

| Logit | Sigmoid(logit) | Discriminator's guess |
|---|---|---|
| Very negative (e.g. -5) | ≈ 0.01 | Confident: FAKE |
| ~0 | ≈ 0.5 | Uncertain |
| Very positive (e.g. +5) | ≈ 0.99 | Confident: REAL |

In [ ]:
# Run the batch through the model
logits = model(batch, training=False).numpy().flatten()
probs = tf.sigmoid(logits).numpy()

print(f'Batch shape:  {batch.shape}   (50 regions in the batch)')
print(f'Output shape: {logits.shape}  (one logit per region)')
print()
print(f'First 10 logits:                     {logits[:10].round(3)}')
print(f'First 10 sigmoid-transformed probs:  {probs[:10].round(3)}')
print()
print(f'Mean logit: {logits.mean():+.4f}')
print(f'Mean prob:  {probs.mean():.4f}')

**The output should be around 0.5.** Every region gets predicted at close to 50/50 probability of being real.

That is exactly what you should expect from an **untrained network**. The weights are still their random initialization from Cell 3, so the network has no idea what real mosquito data looks like. It's guessing.

The GAN training loop (notebook 03, when we get there) is what sharpens the discriminator into actually being able to tell real from fake. During training, the mean output for real regions drifts toward 1.0 and the mean output for fake regions drifts toward 0.0. As the generator (θ) improves, those means converge back toward each other at 0.5. That's the Nash equilibrium we discussed for GAN convergence.

## Cell 8: Sanity check with real simulated data from notebook 01

**One-liner:** hook up the simulator from notebook 01 to the discriminator here. See that the pipeline connects end-to-end.

This confirms that `simulation.dadi_joint_mig`'s output can actually be fed into the CNN without any shape gymnastics. In practice pg-gan's data loader (`real_data_random.py`) does a bit more work (adding the inter-SNP distance channel, slicing to exactly 72 SNPs, batching), but conceptually the tensor shape is what we expect.

In [ ]:
import param_set, simulation

params = param_set.ParamSet(simulation.dadi_joint_mig)
ts = simulation.dadi_joint_mig(params, sample_sizes=[POP1_HAPS, POP2_HAPS],
                                seed=42, reco=1.45e-8)

# Extract the genotype matrix and reshape to the discriminator's expected input
X = ts.genotype_matrix()   # shape (num_SNPs, num_haplotypes)
print(f'Raw msprime genotype matrix: shape {X.shape}')

# Transpose to (haplotypes, SNPs), slice to 72 SNPs, add a fake distance channel
X_haps_first = X.T[:, :NUM_SNPS]                            # (224, 72)
X_2ch = np.stack([X_haps_first, np.ones_like(X_haps_first)], axis=-1)  # (224, 72, 2)
X_batched = X_2ch[np.newaxis].astype(np.float32)             # (1, 224, 72, 2)
print(f'After reshaping for the CNN: shape {X_batched.shape}')

# Feed it through the discriminator
output = model(X_batched, training=False).numpy().flatten()
print(f'\nDiscriminator output (logit):        {output[0]:+.4f}')
print(f'Discriminator output (probability):  {tf.sigmoid(output[0]).numpy():.4f}')
print(f'\nAgain ~0.5 because the network is untrained. But the pipeline works end-to-end.')

## What you now know

- The discriminator is a **CNN with about 666K parameters**, most of them in one big Dense layer.
- It has a **two-branch design**: POP1 and POP2 go through separate copies of the same convolutional layers, then get concatenated at the dense layers. This is a Siamese-style architecture.
- The `(1, 5)` **convolutional kernel** means each filter sees 5 consecutive SNPs of one haplotype at a time. This preserves **permutation invariance across haplotypes**, which is critical because haplotype order in the genotype matrix carries no biological meaning.
- **`ReduceMean`** is the permutation-invariant pooling step that collapses across the haplotype dimension. The paper uses mean rather than sum specifically because POP1 and POP2 have unequal sample sizes (62 vs. 162), and mean normalizes them.
- The final output is a **single logit per region**, not a probability. Apply sigmoid to interpret it as "probability that this region is real".
- The **untrained network outputs ~0.5** for everything, as expected. Actual scoring skill comes from GAN training (next notebook).

**Next up:** notebook 03 will look at the GAN training loop itself, where the discriminator and the θ vector update in alternating steps.